# 05 - Fine-tune the improved MobileNet image branch

Compare the legacy and gated-residual classifier heads using training-signers-only cross-validation. Then fine-tune the final MobileNetV3-Small feature stages with conservative, orientation-preserving augmentation and train the selected head for the improved model. The image/landmark blend and no-hand fallback are selected using validation data only. This stage does not use the test signer or change the baseline checkpoint.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.improved_signer_cv import run_head_cv

config = load_config()
head_summary = run_head_cv(config)
ensemble_heads = head_summary[head_summary['model'].str.endswith('_ensemble')]
if ensemble_heads.empty:
    raise RuntimeError('Head cross-validation did not produce an ensemble candidate.')
selected_head = ensemble_heads.iloc[0]['model'].removesuffix('_ensemble')
print('Cross-validation-selected image head:', selected_head)


In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, project_path
import src.landmark_image_ensemble as landmark_image_ensemble

importlib.reload(landmark_image_ensemble)
config = load_config()
head_summary_path = (
    project_path(config, 'predictions') / 'improved_model_cv' / 'classifier_head_summary.csv'
)
if not head_summary_path.is_file():
    raise FileNotFoundError('Run the training-signer head cross-validation cell first.')
head_summary = pd.read_csv(head_summary_path)
ensemble_heads = head_summary[head_summary['model'].str.endswith('_ensemble')]
if ensemble_heads.empty:
    raise ValueError(f'No ensemble head candidates in {head_summary_path}.')
config['model']['image_head'] = ensemble_heads.iloc[0]['model'].removesuffix('_ensemble')
print('Cross-validation-selected image head:', config['model']['image_head'])
validation_baseline = project_path(config, 'checkpoints') / 'landmark_mlp_validation_best.pt'
outcome = landmark_image_ensemble.run_experiment(
    config, baseline_checkpoint_path=validation_baseline,
)
print(f"Best validation accuracy: {outcome['validation_accuracy']:.2%}")
print(f"Selected image weight: {outcome['image_weight']:.1f}; no-hand image fallback: {outcome['no_hand_image_fallback']}")
print(f"Selected image-head epoch: {outcome['epoch']}")
print('Saved checkpoint:', outcome['checkpoint'])
